<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025S1/blob/main/week12_NLP/word_embedding_customized_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customized word embedding layer

In this lab exercise, you will train your own word embedding model for a sentiment classification task, and then visualize them in the [Embedding Projector](https://projector.tensorflow.org) (shown in the image below). 

<img src="https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/resources/it3103/embedding_projector.png" alt="Screenshot of embedding projector" width="400"/>

### Word embeddings

Word embeddings give us a way to use an efficient, dense representation in which similar words have a similar encoding. An embedding is a dense vector of floating point values (the length of the vector is a hyperparameter). Instead of specifying the embedding matrix manually, they are trainable parameters (weights learned by the model during training, in the same way as a model learns weights for a dense layer). It is common to see word embeddings that are 8-dimensional (for small datasets), up to 1024-dimensions when working with large datasets. A higher dimensional embedding can capture fine-grained relationships between words, but takes more data to learn.

<img src="https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/resources/it3103/embedding2.png" alt="Diagram of an embedding" width="400"/>

Above is a diagram for a word embedding. Each word is represented as a 4-dimensional vector of floating point values. Another way to think of an embedding is as "lookup table". After a embedding matrix (weights) have been learned, you can encode each word into a dense vector as shown in the lookup table.

## Setup

In [1]:
import io
import os
import shutil
import tensorflow as tf

### Download the IMDb Dataset
You will use the [Large Movie Review Dataset](http://ai.stanford.edu/~amaas/data/sentiment/) through the tutorial. You will train a sentiment classifier model on this dataset and in the process, train an embedding layer from scratch. To read more about loading a dataset from scratch, see the [Loading text tutorial](../load_data/text.ipynb).  

Download the dataset using Keras file utility and take a look at the directories.

In [ ]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file("aclImdb_v1.tar.gz", url,
                                    untar=True, cache_dir='.',
                                    cache_subdir='')

# dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb')
dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb_v1_extracted/aclImdb')
os.listdir(dataset_dir)

84125825/84125825 [==============================] - 28s 0us/step


['imdb.vocab', 'imdbEr.txt', 'README', 'test', 'train']

Take a look at the `train/` directory. It has `pos` and `neg` folders with movie reviews labelled as positive and negative respectively. You will use reviews from `pos` and `neg` folders to train a binary classification model.

In [3]:
train_dir = os.path.join(dataset_dir, 'train')
os.listdir(train_dir)
test_dir = os.path.join(dataset_dir, 'test')
os.listdir(test_dir)

['labeledBow.feat', 'neg', 'pos', 'urls_neg.txt', 'urls_pos.txt']

The `train` directory also has additional folders which should be removed before creating training dataset.

In [4]:
remove_dir = os.path.join(train_dir, 'unsup')
shutil.rmtree(remove_dir)

Next, create a `tf.data.Dataset` using `tf.keras.preprocessing.text_dataset_from_directory`. You can read more about this utility from the [api documentation](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text_dataset_from_directory). 

Use the `train` directory to create both train and validation datasets with a split of 20% for validation.

In [5]:
batch_size = 1024
seed = 123
train_ds = tf.keras.preprocessing.text_dataset_from_directory(
    train_dir, batch_size=batch_size, validation_split=0.2, 
    subset='training', seed=seed)
val_ds = tf.keras.preprocessing.text_dataset_from_directory(
    train_dir, batch_size=batch_size, validation_split=0.2, 
    subset='validation', seed=seed)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


Take a look at a few movie reviews and their labels `(1: positive, 0: negative)` from the train dataset.


In [6]:
for text_batch, label_batch in train_ds.take(1):
    for i in range(5):
        print(label_batch[i].numpy(), text_batch.numpy()[i])

0 b"Oh My God! Please, for the love of all that is holy, Do Not Watch This Movie! It it 82 minutes of my life I will never get back. Sure, I could have stopped watching half way through. But I thought it might get better. It Didn't. Anyone who actually enjoyed this movie is one seriously sick and twisted individual. No wonder us Australians/New Zealanders have a terrible reputation when it comes to making movies. Everything about this movie is horrible, from the acting to the editing. I don't even normally write reviews on here, but in this case I'll make an exception. I only wish someone had of warned me before I hired this catastrophe"
1 b'This movie is SOOOO funny!!! The acting is WONDERFUL, the Ramones are sexy, the jokes are subtle, and the plot is just what every high schooler dreams of doing to his/her school. I absolutely loved the soundtrack as well as the carefully placed cynicism. If you like monty python, You will love this film. This movie is a tad bit "grease"esk (without

### Configure the dataset for performance

These are two important methods you should use when loading data to make sure that I/O does not become blocking.

`.cache()` keeps data in memory after it's loaded off disk. This will ensure the dataset does not become a bottleneck while training your model. If your dataset is too large to fit into memory, you can also use this method to create a performant on-disk cache, which is more efficient to read than many small files.

`.prefetch()` overlaps data preprocessing and model execution while training. 

You can learn more about both methods, as well as how to cache data to disk in the [data performance guide](https://www.tensorflow.org/guide/data_performance).

In [7]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Using the Embedding layer

Keras makes it easy to use word embeddings. Take a look at the [Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) layer.

The Embedding layer can be understood as a lookup table that maps from integer indices (which stand for specific words) to dense vectors (their embeddings). The dimensionality (or width) of the embedding is a parameter you can experiment with to see what works well for your problem, much in the same way you would experiment with the number of neurons in a Dense layer.


In [ ]:
import tensorflow as tf


class DenseVectorEmbedding(tf.keras.layers.Layer):
    """
    Embedding implemented *without* tf.gather / tf.nn.embedding_lookup.

    It converts token IDs to one-hot vectors (depth = vocab size) and
    multiplies them by the trainable embedding matrix.

    Args
    ----
    input_dim  : int   vocabulary size (|V|)
    output_dim : int   embedding dimension (D)
    mask_zero  : bool  if True, treat index 0 as padding + propagate a mask
    """

    def __init__(
        self,
        input_dim,
        output_dim,
        mask_zero=False,
        embeddings_initializer="uniform",
        embeddings_regularizer=None,
        embeddings_constraint=None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.input_dim = int(input_dim)
        self.output_dim = int(output_dim)
        self.mask_zero = bool(mask_zero)

        self.emb_init = tf.keras.initializers.get(embeddings_initializer)
        self.emb_reg  = tf.keras.regularizers.get(embeddings_regularizer)
        self.emb_con  = tf.keras.constraints.get(embeddings_constraint)

    # ---------------- weight ---------------- #
    def build(self, _):
        self.embeddings = self.add_weight(
            name="embeddings",
            shape=(self.input_dim, self.output_dim),
            initializer=self.emb_init,
            regularizer=self.emb_reg,
            constraint=self.emb_con,
            trainable=True,
            dtype=self.dtype,
        )
        super().build(_)

    # ---------------- call ------------------ #
    def call(self, inputs):
        """
        inputs: int32 / int64 tensor with any rank (...,)

        Process:
        1. one-hot encode → (..., V)
        2. matrix-multiply with W  → (..., D)
        """
        if not inputs.dtype.is_integer:
            raise ValueError("OneHotEmbedding expects integer indices")

        # (...,) → (..., V)
        one_hot = tf.one_hot(inputs, depth=self.input_dim, dtype=self.embeddings.dtype)

        # einsum or matmul; here tensordot for generality
        output = tf.tensordot(one_hot, self.embeddings, axes=[[ -1 ], [ 0 ]])
        # tensordot places the new axis at the end automatically; shape (..., D)
        return output

    # ------------- mask propagation --------- #
    def compute_mask(self, inputs, mask=None):
        if self.mask_zero:
            return tf.not_equal(inputs, 0)
        return None

    # ------------- shape helpers ------------ #
    def compute_output_shape(self, input_shape):
        return input_shape + (self.output_dim,)

    # ------------- serialization ------------ #
    def get_config(self):
        cfg = super().get_config()
        cfg.update(
            {
                "input_dim": self.input_dim,
                "output_dim": self.output_dim,
                "mask_zero": self.mask_zero,
                "embeddings_initializer":
                    tf.keras.initializers.serialize(self.emb_init),
                "embeddings_regularizer":
                    tf.keras.regularizers.serialize(self.emb_reg),
                "embeddings_constraint":
                    tf.keras.constraints.serialize(self.emb_con),
            }
        )
        return cfg


In [ ]:
embedding_layer = DenseVectorEmbedding(1000, 5)

In [ ]:
# # Embed a 1,000 word vocabulary into 5 dimensions.
# embedding_layer = tf.keras.layers.Embedding(1000, 5)

When you create an Embedding layer, the weights for the embedding are randomly initialized (just like any other layer). During training, they are gradually adjusted via backpropagation. Once trained, the learned word embeddings will roughly encode similarities between words (as they were learned for the specific problem your model is trained on).

If you pass an integer to an embedding layer, the result replaces each integer with the vector from the embedding table:

In [10]:
result = embedding_layer(tf.constant([0,2,111199]))
result.numpy()

array([[-0.02169654, -0.02791127,  0.01373536,  0.04255647, -0.04889686],
       [ 0.04856453, -0.02176286,  0.04883268,  0.04513334,  0.03265443],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]],
      dtype=float32)

For text or sequence problems, the Embedding layer takes a 2D tensor of integers, of shape `(samples, sequence_length)`, where each entry is a sequence of integers. It can embed sequences of variable lengths. You could feed into the embedding layer above batches with shapes `(32, 10)` (batch of 32 sequences of length 10) or `(64, 15)` (batch of 64 sequences of length 15).

The returned tensor has one more axis than the input, the embedding vectors are aligned along the new last axis. Pass it a `(2, 3)` input batch and the output is `(2, 3, N)`


In [11]:
result = embedding_layer(tf.constant([[0,1,2],[3,4,5]]))
result.shape

TensorShape([2, 3, 5])

When given a batch of sequences as input, an embedding layer returns a 3D floating point tensor, of shape `(samples, sequence_length, embedding_dimensionality)`. To convert from this sequence of variable length to a fixed representation there are a variety of standard approaches. You could use an RNN (which will be covered in the subsequent lesson), or pooling layer before passing it to a Dense layer. This lab uses pooling because it's the simplest.

## Text preprocessing

Before we feed the text to our embedding layer, we need to first break up our text string into an array of numbers. The number is then used as index into the lookup table of Embedding layer to produce the corresponding word embedding. This is the job of text tokenizer. 

TextVectorization layer is a text tokenizer which breaks up the text into words (it is similar to Keras Tokenizer but implemented as a layer). You can read more about TextVectorization layer [here](https://www.tensorflow.org/api_docs/python/tf/keras/layers/experimental/preprocessing/TextVectorization).


In [12]:
# Vocabulary size and number of words in a sequence.
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 200

# Use the text vectorization layer to normalize, split, and map strings to 
# integers. 
# We also set output_sequence length so that longer sentence will be truncated 
# and shorter sentence will be padded
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_ds = train_ds.map(lambda x, y: x)
vectorize_layer.adapt(text_ds)

In [13]:
vectorize_layer.get_vocabulary()[:50]

['',
 '[UNK]',
 'the',
 'and',
 'a',
 'of',
 'to',
 'is',
 'in',
 'it',
 'i',
 'this',
 'that',
 'br',
 'was',
 'as',
 'with',
 'for',
 'movie',
 'but',
 'film',
 'on',
 'not',
 'you',
 'are',
 'his',
 'have',
 'be',
 'he',
 'one',
 'its',
 'at',
 'all',
 'by',
 'an',
 'they',
 'who',
 'from',
 'so',
 'like',
 'her',
 'just',
 'or',
 'about',
 'has',
 'if',
 'out',
 'some',
 'there',
 'what']

Let us try to use our TextVectorization layer to tokenize some sample text. TextVectorization layer expects a list of string instead of a single string.

Note that the output for the third word 'Singapore', the token assigned is 1, which is 'UNK' token.  This is because the word cannot be found in the training vocab. 

In [14]:
sample_text = ['I love Singapore!']
print(vectorize_layer(sample_text))

# Print out corresponding word for token index 116
print(vectorize_layer.get_vocabulary()[116])

tf.Tensor(
[[ 10 117   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0]], shape=(1, 200), dtype=int64)
did


## Training Embedding

In the lecture, we know there are two ways to obtain the text embedding: one is to train it together with our ML task (e.g. text classification task), another is to use pre-trained embedding like GloVe, Word2Vec, etc.

We will look at how we can train our own embedding first by jointly train it with our sentiment classification task.

## Create a classification model

Use the [Keras Sequential API](../../guide/keras) to define the sentiment classification model. In this case it is a "Continuous bag of words" style model.
* The [`TextVectorization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/experimental/preprocessing/TextVectorization) layer transforms strings into vocabulary indices. You have already initialized `vectorize_layer` as a TextVectorization layer and built it's vocabulary by calling `adapt` on `text_ds`. Now vectorize_layer can be used as the first layer of your end-to-end classification model, feeding tranformed strings into the Embedding layer.
* The [`Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) layer takes the integer-encoded vocabulary and looks up the embedding vector for each word-index. These vectors are learned as the model trains. The vectors add a dimension to the output array. The resulting dimensions are: `(batch, sequence, embedding)`.

* The [`GlobalAveragePooling1D`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalAveragePooling1D) layer returns a fixed-length output vector for each example by averaging over the sequence dimension. This allows the model to handle input of variable length, in the simplest way possible.

* The fixed-length output vector is piped through a fully-connected ([`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)) layer with 16 hidden units.

* The last layer is densely connected with a single output node. 

Caution: This model doesn't use masking, so the zero-padding is used as part of the input and may affect the model performance. Some layers such as RNN is mask-aware layer, and will be able to ignore those zero-padded portion of the input when computing the loss.

In [ ]:
EMBEDDING_DIM=128

model = tf.keras.Sequential([
    vectorize_layer,
    DenseVectorEmbedding(VOCAB_SIZE, EMBEDDING_DIM, name='embedding'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
# EMBEDDING_DIM=128

# model = tf.keras.Sequential([
#     vectorize_layer,
#     tf.keras.layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, name='embedding'),
#     tf.keras.layers.GlobalAveragePooling1D(),
#     tf.keras.layers.Dense(32, activation='relu'),
#     tf.keras.layers.Dropout(0.5),
#     tf.keras.layers.Dense(8, activation='relu'),
#     tf.keras.layers.Dropout(0.5),
#     tf.keras.layers.Dense(1, activation='sigmoid')
# ])

## Compile and train the model

You will use [TensorBoard](https://www.tensorflow.org/tensorboard) to visualize metrics including loss and accuracy. Create a `tf.keras.callbacks.TensorBoard`.

In [16]:
root_logdir = os.path.join(os.curdir, "tb_logs")

def get_run_logdir():    # use a new directory for each run
	import time
	run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
	return os.path.join(root_logdir, run_id)

run_logdir = get_run_logdir()
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=run_logdir)
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="bestcheckpoint.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

Compile and train the model using the `Adam` optimizer and `BinaryCrossentropy` loss. 

In [17]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=['accuracy'])

In [18]:
model.fit(
    train_ds,
    validation_data=val_ds, 
    epochs=10,
    callbacks=[tensorboard_callback, model_checkpoint_callback])

Epoch 1/10
20/20 [==============================] - 102s 5s/step - loss: 0.6915 - accuracy: 0.5277 - val_loss: 0.6880 - val_accuracy: 0.5116
Epoch 2/10
20/20 [==============================] - 87s 4s/step - loss: 0.6811 - accuracy: 0.5960 - val_loss: 0.6697 - val_accuracy: 0.6406
Epoch 3/10
20/20 [==============================] - 94s 5s/step - loss: 0.6529 - accuracy: 0.6808 - val_loss: 0.6276 - val_accuracy: 0.7360
Epoch 4/10
20/20 [==============================] - 102s 5s/step - loss: 0.6046 - accuracy: 0.7544 - val_loss: 0.5662 - val_accuracy: 0.7892
Epoch 5/10
20/20 [==============================] - 109s 6s/step - loss: 0.5439 - accuracy: 0.8089 - val_loss: 0.5035 - val_accuracy: 0.8194
Epoch 6/10
20/20 [==============================] - 108s 5s/step - loss: 0.4842 - accuracy: 0.8367 - val_loss: 0.4472 - val_accuracy: 0.8514
Epoch 7/10
20/20 [==============================] - 106s 5s/step - loss: 0.4328 - accuracy: 0.8601 - val_loss: 0.4083 - val_accuracy: 0.8614
Epoch 8/10
20/2

Visualize the model metrics in TensorBoard.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

With this approach the model reaches a validation accuracy of around 86%.

Note: Your results may be a bit different, depending on how weights were randomly initialized before training the embedding layer. 


Let's evaluate the model on our test dataset.

In [19]:
test_ds = tf.keras.preprocessing.text_dataset_from_directory(
    test_dir, 
    batch_size=batch_size)

Found 25000 files belonging to 2 classes.


In [20]:
model.load_weights("bestcheckpoint.weights.h5")
test_loss, test_acc = model.evaluate(test_ds)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

25/25 [==============================] - 95s 3s/step - loss: 0.3875 - accuracy: 0.8531
Test Loss: 0.38752481341362
Test Accuracy: 0.8530799746513367


Let's go ahead and save our model. You will see that our model achieve an accuracy of around 82%. 

In [21]:
model.save('sentiment_model.keras')